# BOT de Ventas — Agente Pandas + RAG + Mistral
Corpus: ventas_normalizado.csv | Embeddings: HuggingFace | VectorDB: ChromaDB | LLM: Mistral

In [1]:
# BLOQUE 1 — Instalacion de dependencias
!pip install -q langchain langchain-community langchain-mistralai langchain-experimental
!pip install -q chromadb sentence-transformers
!pip install -q pandas tabulate langchain-core

print("Todas las dependencias instaladas")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [2]:
# BLOQUE 2 — Imports
import os
import pandas as pd
from google.colab import files, userdata

from langchain_mistralai import ChatMistralAI
from langchain_community.document_loaders import DataFrameLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_experimental.agents import create_pandas_dataframe_agent

print("Imports correctos")

/tmp/ipykernel_1103/204473239.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DataFrameLoader
/tmp/ipykernel_1103/204473239.py:13: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.agents import create_pandas_dataframe_agent


Imports correctos


In [12]:
# BLOQUE 3 — API Key y carga del dataset
import os
from google.colab import files, userdata

os.environ["MISTRAL_API_KEY"] = userdata.get('bot-ventas-rag')
key = os.environ["MISTRAL_API_KEY"]
print(f"API Key cargada: {key[:8]}...")

print("Subi tu archivo ventas_normalizado.csv:")
uploaded = files.upload()
nombre_csv = list(uploaded.keys())[0]
df = pd.read_csv(nombre_csv, encoding='latin-1')

print(f"Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head(3)

API Key cargada: 9JUL5HFY...
Subi tu archivo ventas_normalizado.csv:


Saving ventas_normalizado.csv to ventas_normalizado (2).csv
Dataset cargado: 2823 filas x 24 columnas


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,PHONE,ADDRESSLINE1,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30,95.70,2,2871.00,2003-02-24,SHIPPED,1,2,2003,...,2125557818,897 Long Airport Avenue,NYC,NY,10022,USA,UNKNOWN,Yu,Kwai,SMALL
1,10121,34,81.35,5,2765.90,2003-05-07,SHIPPED,2,5,2003,...,26.47.1555,59 rue de l'Abbaye,Reims,UNKNOWN,51100,FRANCE,EMEA,Henriot,Paul,SMALL
2,10134,41,94.74,2,3884.34,2003-07-01,SHIPPED,3,7,2003,...,+33 1 46 62 7555,27 rue du Colonel Pierre Avia,Paris,UNKNOWN,75508,FRANCE,EMEA,Da Cunha,Daniel,MEDIUM


In [13]:
# BLOQUE 4 — Inicializar el LLM Mistral
llm = ChatMistralAI(
    model="mistral-small-latest",
    temperature=0.2,
    api_key=os.environ["MISTRAL_API_KEY"]
)

test = llm.invoke("Responde en una linea: estas listo para analizar datos?")
print("Test de conexion con Mistral:")
print(test.content)

Test de conexion con Mistral:
¡Sí, listo para analizar datos! 🚀


In [14]:
# BLOQUE 5 — Creacion del Agente Pandas
# El agente genera y ejecuta codigo Python sobre el DataFrame
# para responder preguntas numericas y estadisticas

prefix_prompt = """
Eres un asistente experto en analisis de datos de ventas.
Tenes acceso a un DataFrame de pandas llamado df con todas las columnas del dataset.

Instrucciones criticas:
1. Para cualquier pregunta numerica o estadistica, DEBES generar y ejecutar codigo Python.
2. Si te preguntan por ventas totales, usa la columna sales.
3. Si hay valores nulos o unknown, ignoralos en calculos matematicos.
4. Responde siempre en espanol de forma clara y concisa.
"""

agent = create_pandas_dataframe_agent(
    llm,
    df,
    prefix=prefix_prompt,
    verbose=True,
    allow_dangerous_code=True,
    agent_type="tool-calling"
)

print("Agente Pandas creado correctamente")
print(f"Herramientas: {[t.name for t in agent.tools]}")

Agente Pandas creado correctamente
Herramientas: ['python_repl_ast']


In [15]:
# BLOQUE 6 — Convertir todas las columnas del CSV a documentos de texto
# fila_a_texto() toma TODAS las columnas automaticamente sin hardcodear nada

def fila_a_texto(row):
    partes = []
    for col in row.index:
        valor = row[col]
        if pd.notna(valor):
            partes.append(f"{col}: {valor}")
    return ". ".join(partes) + "."

df['resumen_fila'] = df.apply(fila_a_texto, axis=1)

print("Ejemplo de fila convertida a texto:")
print(df['resumen_fila'].iloc[0])

loader = DataFrameLoader(df, page_content_column='resumen_fila')
documentos = loader.load()
print(f"\nTotal de documentos indexados: {len(documentos)}")

Ejemplo de fila convertida a texto:
ORDERNUMBER: 10107. QUANTITYORDERED: 30. PRICEEACH: 95.7. ORDERLINENUMBER: 2. SALES: 2871.0. ORDERDATE: 2003-02-24. STATUS: SHIPPED. QTR_ID: 1. MONTH_ID: 2. YEAR_ID: 2003. PRODUCTLINE: MOTORCYCLES. MSRP: 95. PRODUCTCODE: S10_1678. CUSTOMERNAME: LAND OF TOYS INC.. PHONE: 2125557818. ADDRESSLINE1: 897 Long Airport Avenue. CITY: NYC. STATE: NY. POSTALCODE: 10022. COUNTRY: USA. TERRITORY: UNKNOWN. CONTACTLASTNAME: Yu. CONTACTFIRSTNAME: Kwai. DEALSIZE: SMALL.

Total de documentos indexados: 2823


In [16]:
# BLOQUE 7 — Embeddings con HuggingFace + Base vectorial ChromaDB
# HuggingFace es gratuito y corre local en Colab, no necesita API key
# ChromaDB guarda los vectores en RAM para busqueda por similitud

print("Transformando el corpus a vectores semanticos...")
embeddings_local = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vector_db = Chroma.from_documents(documentos, embeddings_local)
retriever = vector_db.as_retriever(search_kwargs={"k": 5})

print("Sistema RAG indexado y listo!")

Transformando el corpus a vectores semanticos...


/tmp/ipykernel_1103/1121579090.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings_local = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sistema RAG indexado y listo!


In [18]:
# BLOQUE 8 — Construir la cadena RAG con LCEL
# Flujo: Pregunta -> RAG (ChromaDB) -> Prompt con contexto -> Mistral -> Respuesta

prompt_template = PromptTemplate.from_template("""Eres un analista experto en ventas. Tenes acceso a un corpus
de datos de ventas reales. Responde UNICAMENTE basandote en el contexto
proporcionado. Si la informacion no esta en el contexto, di exactamente:
"No encontre esa informacion en los datos disponibles."

Responde siempre en espanol, de forma clara y concisa.

Contexto del corpus de ventas:
{context}

Pregunta del usuario:
{question}

Respuesta del analista:""")

def formatear_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

cadena_rag = (
    {
        "context": retriever | formatear_docs,
        "question": RunnablePassthrough()
    }
    | prompt_template
    | llm
    | StrOutputParser()
)

print("Cadena RAG construida correctamente.")
print("Flujo: Pregunta -> RAG (corpus) -> Mistral -> Respuesta")

Cadena RAG construida correctamente.
Flujo: Pregunta -> RAG (corpus) -> Mistral -> Respuesta


In [19]:
# BLOQUE 9 — Funcion preguntar combinada (Agente + RAG)
# El RAG aporta contexto semantico, el agente ejecuta calculos exactos

def preguntar(pregunta: str) -> str:
    print(f"\n{'='*55}")
    print(f"Pregunta: {pregunta}")
    print(f"{'='*55}\n")

    # 1. Recuperar contexto relevante del RAG
    docs = retriever.invoke(pregunta)
    contexto = formatear_docs(docs)

    # 2. Armar pregunta enriquecida con contexto para el agente
    pregunta_con_contexto = f"""
Contexto relevante del dataset:
{contexto}

Pregunta: {pregunta}
"""

    # 3. El agente ejecuta codigo sobre el DataFrame y responde
    resultado = agent.invoke({"input": pregunta_con_contexto})
    respuesta = resultado["output"]

    print(f"\n{'─'*55}")
    print(f"Respuesta:\n{respuesta}")
    print(f"{'─'*55}\n")

    return respuesta

print("Funcion preguntar() lista — Agente + RAG combinados")

Funcion preguntar() lista — Agente + RAG combinados


In [20]:
# BLOQUE 10 — Pruebas de funcionamiento
preguntar("cuantas filas tiene el dataset?")


Pregunta: cuantas filas tiene el dataset?



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': 'len(df)'}`


2823El dataset tiene **2823 filas**.

> Finished chain.

───────────────────────────────────────────────────────
Respuesta:
El dataset tiene **2823 filas**.
───────────────────────────────────────────────────────



'El dataset tiene **2823 filas**.'

In [21]:
preguntar("cual es la linea de producto mas vendida?")


Pregunta: cual es la linea de producto mas vendida?



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "ventas_por_linea = df.groupby('PRODUCTLINE')['SALES'].sum().sort_values(ascending=False)\nventas_por_linea"}`


PRODUCTLINE
CLASSIC CARS        3919615.66
VINTAGE CARS        1903150.84
MOTORCYCLES         1166388.34
TRUCKS AND BUSES    1127789.84
PLANES               975003.57
SHIPS                714437.13
TRAINS               226243.47
Name: SALES, dtype: float64La línea de producto más vendida es **CLASSIC CARS**, con un total de ventas de **3,919,615.66**.

> Finished chain.

───────────────────────────────────────────────────────
Respuesta:
La línea de producto más vendida es **CLASSIC CARS**, con un total de ventas de **3,919,615.66**.
───────────────────────────────────────────────────────



'La línea de producto más vendida es **CLASSIC CARS**, con un total de ventas de **3,919,615.66**.'

In [22]:
preguntar("cuales son los 5 paises con mas ventas?")


Pregunta: cuales son los 5 paises con mas ventas?



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "ventas_por_pais = df.groupby('COUNTRY')['SALES'].sum().sort_values(ascending=False).head(5)\nventas_por_pais"}`


COUNTRY
USA          3627982.83
SPAIN        1215686.92
FRANCE       1110916.52
AUSTRALIA     630623.10
UK            478880.46
Name: SALES, dtype: float64Los 5 países con más ventas son:

1. **USA**: 3,627,982.83
2. **SPAIN**: 1,215,686.92
3. **FRANCE**: 1,110,916.52
4. **AUSTRALIA**: 630,623.10
5. **UK**: 478,880.46

> Finished chain.

───────────────────────────────────────────────────────
Respuesta:
Los 5 países con más ventas son:

1. **USA**: 3,627,982.83
2. **SPAIN**: 1,215,686.92
3. **FRANCE**: 1,110,916.52
4. **AUSTRALIA**: 630,623.10
5. **UK**: 478,880.46
───────────────────────────────────────────────────────



'Los 5 países con más ventas son:\n\n1. **USA**: 3,627,982.83\n2. **SPAIN**: 1,215,686.92\n3. **FRANCE**: 1,110,916.52\n4. **AUSTRALIA**: 630,623.10\n5. **UK**: 478,880.46'

In [ ]:
# BLOQUE 11 — Chat interactivo
print("=" * 55)
print("  BOT ANALISTA DE VENTAS — Agente Pandas + RAG + Mistral")
print("  Escribi 'salir' para terminar la sesion.")
print("=" * 55)

while True:
    pregunta = input("\nQue queres saber sobre las ventas?\n> ").strip()

    if not pregunta:
        continue

    if pregunta.lower() == "salir":
        print("\nSesion finalizada. Hasta la proxima!")
        break

    try:
        preguntar(pregunta)
    except Exception as e:
        print(f"\nError: {e}")
        print("Verifica que la API key de Mistral sea valida y que el RAG este inicializado.")

  BOT ANALISTA DE VENTAS — Agente Pandas + RAG + Mistral
  Escribi 'salir' para terminar la sesion.

Que queres saber sobre las ventas?
> pais que compra mas

Pregunta: pais que compra mas



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "df['COUNTRY'].value_counts().idxmax()"}`


USAEl país que realiza más compras en el dataset es **USA**.

> Finished chain.

───────────────────────────────────────────────────────
Respuesta:
El país que realiza más compras en el dataset es **USA**.
───────────────────────────────────────────────────────

